In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
from scipy import stats # for Box-Cox Tranformer
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import PowerTransformer

In [ ]:
# STEPS in Building ML Model:
## Get Data + Try To Understand Data and Relation Between Varaibles
## Feature Enegnering (30%)
## Feature Selection
## Model Creationg
## HyperParamter Tunning
## Model Deployments

## Feature Engineering

- **EDA**:
  - numerical features
  - categorical features
  - missing values
  - outliers
- **Handling missing values**
  - 1. mean, 2. median, 3. mode
- Handling unbalanced dataset
- Treating the outliers
- Scaling the Data (Standardisation and normalisation)
- Convert The Categorical Feature Into Numerical Features
- **Feature Selection**
  - Correlation
  - K neighbors
  - Features Importances
  - ....

### Get Data + EDA


In [ ]:
path = kagglehub.dataset_download("abdellahkarani/renewable")
print(path)

In [ ]:
df = pd.read_csv("/kaggle/input/datasets/abdellahkarani/renewable/Renewable.csv")
df.head(3)

In [ ]:
'''
    Feature Description:
        - Energy delta [WH]: Target Varaible
        - GHI (Irradiance): كمية الضو ديال الشمس لي كتوصل للأرض
            كلما طالع = طاقة أكثر
        - isSun: 0: night, 1: day
        - sunlightTime: time thst sun exit
        - dayLength:
        - SunlightTime/daylength: clean day or a little there is a little of cloud
            the values almost 1 --> clean day
        - clouds_all: percentage of clouds
            more clouds less energy
        - weather_type: type of weather
            0, 1, 2, 3, 4
        - humidity: ....
            low humidity = high energey
        - rain_1h: rain in last one hour
            affect energy
        - snow_1h: snow in last one hour
            affect a lot energy
        - temp
            hight temp = high energy
        - wind_speed
        - pressure
        - hour
        - month
'''
# df.nunique() # the values show us that there is more stable solar energy
# df["SunlightTime/daylength"].iloc[0: 1300]
# df["clouds_all"].iloc[400: 3490]
# df["temp"].iloc[400: 3490]
# df["weather_type"].iloc[400: 3490]
# df["Energy delta[Wh]"].iloc[400: 3490]
# df["humidity"].iloc[400: 3490]
# df["rain_1h"].iloc[0: 100]
# df["hour"].iloc[0:100]
# df["snow_1h"].unique()
# df["isSun"].iloc[0: 133]

# df.nunique()
df.describe()
# df.info()
# df.isnull().sum() # not null value in dataset

In [ ]:
df.head(2)

In [ ]:
# Correlation Between Varaible
df = df.drop(["Time"], axis=1)

In [ ]:
df.corr()

In [ ]:
# Visual the Correlation
sns.heatmap(df.corr(), annot=True)
plt.rcParams["figure.figsize"] = (12, 10)
plt.show()

In [ ]:
# Group The Data by IsSun and Month and weather_type
# df["IsSun"].unique()
# df["month"].unique()
df0 = df.groupby("weather_type").mean()
df2 = df.groupby("month").mean()
df1 = df.groupby("isSun").mean()
df1

In [ ]:
# Visual Energy delta[Wh] values per month
df2["Energy delta[Wh]"].plot(kind="bar", figsize=(12,6))

plt.xlabel("Month")
plt.ylabel("Energy delta[Wh]")
plt.title("Energy delta[Wh] per month")
plt.show()

In [ ]:
# Problem Here: is that the energy that we produce in night is 0
values = df["Energy delta[Wh]"].where(df["Energy delta[Wh]"] == 0, other=-1).to_numpy()
lenght = (len([n for n in values if n == 0]))
print(df["Energy delta[Wh]"].count())
print(lenght)
print(df["Energy delta[Wh]"].count() - lenght)

In [ ]:
df[np.array(df["isSun"] == 0)]
# Visual Energy delta[Wh] values in Day vs Night
df1["Energy delta[Wh]"].plot(kind="bar", figsize=(12,6))

plt.xlabel("1 - Day vs 0 - Night")
plt.ylabel("Energy delta[Wh]")
plt.title("Energy delta[Wh] per month")
plt.show()

### Other Parts

In [ ]:
# Handling Unblanced Dataset
'''
  - in Classification problem:
    means one class dominate to all samples
      --> 95% “No Fraud” && 5% “Fraud”
  - in Regression problem:
    - We can see a skewed target distribution. where most values are concentrated in one region.
    - problme happend with target varaible
'''

In [ ]:
df.head(8)

## Clean The Data

In [ ]:
# Data Already Cleaned
sum(df.duplicated()) # 0 repetitive data
df_clean = df.drop_duplicates()

In [ ]:
# Extract SubDatset where Energy is Deffrent to Zero
df_filtred = df_clean[df_clean["Energy delta[Wh]"] != 0].reset_index()
df_filtred.head(3)

## Find Outliers

In [ ]:
# Example
'''
    [20, 22, 21, 23, 19, 120]
        --> most values arround 20-23
        --> but 120 is far --> this is an outlier
'''

In [ ]:
df_filtred["Energy delta[Wh]"].describe()

In [ ]:
df_filtred["Energy delta[Wh]"].describe()
# Get The Quartiles
Q1 = np.percentile(df_filtred["Energy delta[Wh]"], 25)
Q2 = np.percentile(df_filtred["Energy delta[Wh]"], 50)
Q3 = np.percentile(df_filtred["Energy delta[Wh]"], 75)

# Calcute The IQR
IQR = Q3 - Q1

# Get Upper and Lower Limit
lower_limit = Q1 - (1.5 * IQR)
uper_limit = Q3 + (1.5 * IQR)

lower_limit, uper_limit

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))

sns.boxplot(x=df_filtred["Energy delta[Wh]"])

plt.title("Boxplot of Energy delta[Wh]")
plt.show()

In [ ]:
# Delete The Outliers
np.where(df >= lower_limit and df <= uper_limit)

## Anaylse the Skewness and Kurtosis + Apply Tranformation

In [ ]:
# Energy delta[Wh] --> Positive Skeweens --> Log Transformation
# Apply Log Tranformation
df_filtred["Energy delta[Wh]"] = np.sqrt(df_filtred["Energy delta[Wh]"])

# Display The New Distrbution
sns.histplot(df_filtred["Energy delta[Wh]"], bins=30)
mean_value = df_filtred["Energy delta[Wh]"].mean()
print("The Skewness: ", df_filtred["Energy delta[Wh]"].skew())
plt.axvline(mean_value, color='red', linestyle='--', linewidth=2, label=f'Median = {mean_value:.2f}')

In [ ]:
sum(df_filtred["GHI"] == 0)

In [ ]:
# GHI --> Right Skeweenes --> Log Transformation
df_filtred["GHI"] = np.sqrt(df_filtred["GHI"])
sns.histplot(df_filtred["GHI"], bins=30)
mean_value = df_filtred["GHI"].mean()
plt.axvline(mean_value, color='red', linestyle='--', linewidth=2,
            label=f'Median = {mean_value:.2f}')

In [ ]:
# temp --> Normal Distribution
sns.histplot(df_filtred["temp"], bins=30)
mean_value = df_filtred["temp"].mean()
plt.axvline(mean_value, color='red', linestyle='--', linewidth=2, label=f'Median = {mean_value:.2f}')

In [ ]:
# pressure --> Normal Distribution
sns.histplot(df_filtred["pressure"], bins=30)
mean_value = df_filtred["pressure"].mean()
plt.axvline(mean_value, color='red', linestyle='--', linewidth=2, label=f'Median = {mean_value:.2f}')

In [ ]:
# humidity --> Left Skewness --> Box-Cox Transform
# Box-Cox cannot handle zeros or negative values.
df_filtred["humidity"] = stats.boxcox(df_filtred["humidity"])[0]
sns.histplot(df_filtred["humidity"], bins=30)

mean_value = df_filtred["humidity"].mean()
plt.axvline(mean_value, color='red', linestyle='--', linewidth=2, label=f'Median = {mean_value:.2f}')

In [ ]:
# wind_speed --> Right Skewness --> Log transformation
df_filtred["wind_speed"] = np.log1p(df_filtred["wind_speed"])
sns.histplot(df_filtred["wind_speed"], bins=30)
mean_value = df_filtred["wind_speed"].mean()
plt.axvline(mean_value, color='red', linestyle='--', linewidth=2, label=f'Median = {mean_value:.2f}')

In [ ]:
# snow_1 --> Right Skewness --> Log transformation
# 

In [ ]:
sum(df_filtred["sunlightTime"] == 0)

In [ ]:
# sunlightTime --> Right Skewness --> Log Tranformation
pt = PowerTransformer(method="yeo-johnson")
df_filtred["sunlightTime"] = pt.fit_transform(df_filtred[["sunlightTime"]])
sns.histplot(df_filtred["sunlightTime"], bins=30)
mean_value = df_filtred["sunlightTime"].mean()
plt.axvline(mean_value, color='red', linestyle='--', linewidth=2, label=f'Median = {mean_value:.2f}')

In [ ]:
# dayLength --> Yeo-Johnson Tranformation
df_filtred["dayLength"] = pt.fit_transform(df_filtred[["dayLength"]])
sns.histplot(df_filtred["dayLength"], bins=30)
mean_value = df_filtred["dayLength"].mean()
plt.axvline(mean_value, color='red', linestyle='--', linewidth=2, label=f'Median = {mean_value:.2f}')

In [ ]:
sum(df["SunlightTime/daylength"] == 0)

In [ ]:
# SunlightTime/daylength --> Log Tranformation
df_filtred["SunlightTime/daylength"] = np.log1p(df_filtred["SunlightTime/daylength"])
sns.histplot(df_filtred["SunlightTime/daylength"], bins=30)
mean_value = df_filtred["SunlightTime/daylength"].mean()
plt.axvline(mean_value, color='red', linestyle='--', linewidth=2, label=f'Median = {mean_value:.2f}')

In [ ]:
# Feature Selection:
'''
    - Feature With High correlation with target varaible are selected.
        --> sometime we find some features usefull and no linear. --> is not good way to based on just correlation
    -
    src: https://www.youtube.com/watch?v=tqpIyvCl4fI
'''

In [ ]:
# Is that Data Lineare or NO